In [ ]:
import warp as wp, numpy as np
from socu._mykernel import MyKernel, device_ptr
wp.init()
k=MyKernel()
B,M,n=1,2,3
E=wp.from_numpy(np.random.randn(B*M,n,n).astype(np.float32), dtype=wp.float32, device='cuda')
D=wp.from_numpy(np.random.randn(B*M,n,n).astype(np.float32), dtype=wp.float32, device='cuda')
s=k.create_stream()
k.syrk_update(device_ptr(E), device_ptr(D), B,M,n, s)
k.stream_synchronize(s)
k.destroy_stream(s)
print('ok', float(D.numpy()[0,0,0]))
display(D.numpy())

ok -2.4601175785064697


array([[[-2.4601176 ,  0.84150386,  1.9434307 ],
        [ 4.100215  , -2.138984  , -6.223983  ],
        [ 1.4693605 , -4.1294546 , -7.060731  ]],

       [[-3.7901633 ,  1.3037043 ,  0.9113276 ],
        [ 0.5584723 , -4.8098574 ,  0.88297856],
        [ 5.1570873 , -0.03931612, -1.3328245 ]]], dtype=float32)

In [1]:
import warp as wp, numpy as np
from socu._mykernel import MyKernel, device_ptr

wp.init()
k = MyKernel()

rng = np.random.default_rng(42)
B, M, n = 100, 21, 400
batch_size = B * M

# 用 float64 生成/计算参考值（高精度）
E0_64 = rng.standard_normal((batch_size, n, n), dtype=np.float64)
D0_64 = rng.standard_normal((batch_size, n, n), dtype=np.float64)

# 当前 CUDA kernel 的接口是 float*，所以送入 GPU 的仍需是 float32
E0 = E0_64.astype(np.float32)
D0 = D0_64.astype(np.float32)

E = wp.from_numpy(E0, dtype=wp.float32, device="cuda")
D = wp.from_numpy(D0, dtype=wp.float32, device="cuda")

s = k.create_stream()
k.syrk_update(device_ptr(E), device_ptr(D), B, M, n, s)
k.stream_synchronize(s)
k.destroy_stream(s)

D_cuda = D.numpy()
print("ok", float(D_cuda[0, 0, 0]))

# NumPy float64 参考：批量矩阵乘法 (batch, n, n) @ (batch, n, n).T
D_ref64 = D0_64 - (E0_64 @ np.swapaxes(E0_64, -1, -2))
D_ref32 = D_ref64.astype(np.float32)

# 对 float32 参考做 allclose（更公平：同一 dtype）
diff32 = D_cuda - D_ref32
max_abs_diff32 = np.max(np.abs(diff32))
max_rel_diff32 = np.max(np.abs(diff32) / (np.abs(D_ref32) + 1e-12))
ok32 = np.allclose(D_cuda, D_ref32, atol=1e-5, rtol=1e-5)

# 同时报告相对 float64 参考的误差（反映 float32 舍入/累加误差）
diff64 = D_cuda.astype(np.float64) - D_ref64
max_abs_diff64 = np.max(np.abs(diff64))
max_rel_diff64 = np.max(np.abs(diff64) / (np.abs(D_ref64) + 1e-18))

print("allclose vs D_ref32:", ok32)
print("max_abs_err vs D_ref32:", float(max_abs_diff32))
print("max_rel_err vs D_ref32:", float(max_rel_diff32))
print("max_abs_err vs D_ref64:", float(max_abs_diff64))
print("max_rel_err vs D_ref64:", float(max_rel_diff64))
print("sample D_cuda[0,0,0]:", float(D_cuda[0, 0, 0]))
print("sample D_ref32[0,0,0]:", float(D_ref32[0, 0, 0]))
print("sample D_ref64[0,0,0]:", float(D_ref64[0, 0, 0]))

# 需要 notebook 环境才有 display；不确定环境时用 print 即可
try:
    from IPython.display import display
    display(D_ref64)
except Exception:
    print(D_ref64[0])

Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "Intel64 Family 6 Model 183 Stepping 1, GenuineIntel"
     "cuda:0"   : "NVIDIA GeForce RTX 4060 Laptop GPU" (8 GiB, sm_89, mempool enabled)
   Kernel cache:
     \\?\C:\Users\haoha\AppData\Local\NVIDIA\warp\Cache\1.11.0
ok -363.03009033203125
allclose vs D_ref32: True
max_abs_err vs D_ref32: 6.103515625e-05
max_rel_err vs D_ref32: 64.911865234375
max_abs_err vs D_ref64: 5.617469446406176e-05
max_rel_err vs D_ref64: 64.9158949856078
sample D_cuda[0,0,0]: -363.03009033203125
sample D_ref32[0,0,0]: -363.0300598144531
sample D_ref64[0,0,0]: -363.0300694701375


array([[[-3.63030069e+02,  1.44986973e+01,  3.54202406e+01, ...,
          2.56751023e+01,  4.74663814e+01,  1.11040707e+01],
        [ 1.68892486e+01, -4.13527120e+02,  1.50602889e+01, ...,
         -8.91856137e-01, -6.50250085e+00, -6.69855555e+00],
        [ 3.42466675e+01,  1.34287497e+01, -3.90867557e+02, ...,
          1.53632132e+01,  1.09496352e+01,  3.99288153e+01],
        ...,
        [ 2.45046996e+01, -7.17974500e-01,  1.31434606e+01, ...,
         -4.08355802e+02, -2.88403661e+01,  1.46997247e+01],
        [ 4.85999493e+01, -6.22381373e+00,  1.20135147e+01, ...,
         -2.74786910e+01, -4.17933881e+02,  4.64782020e+00],
        [ 1.06477683e+01, -6.71176838e+00,  3.76670518e+01, ...,
          1.50199799e+01,  2.45623021e+00, -4.02751293e+02]],

       [[-4.43438479e+02, -4.18181507e+01, -2.00995143e+00, ...,
         -9.96864466e+00,  1.80425967e+01, -3.34721584e+00],
        [-3.94412347e+01, -3.92435870e+02, -3.08863898e+00, ...,
         -4.65571277e+01,  1.79825790e